In [17]:
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import ratio
import re
import src_any_abogados_fuzzymerge_df_er_v1_sec as src
import src_any_abogados_numbersmerge_df_er_v1_sec as src2
#import src_any_abogados_emailmerge_df_er_v1_sec as src3
import src_any_abogados_test_df_er_v1_sec as src4

In [18]:
pd.set_option("display.max_rows", None)

In [19]:
def remove_empty_columns(df):

    df_aux = df.copy()
    empty_columns = list(df_aux.columns[df_aux.isna().mean() == 1])   # Almacena en una lista las columnas que están totalmente vacias
    if "Unnamed: 0" in df_aux.columns:
        empty_columns.append("Unnamed: 0")
    df_aux = df_aux.drop(columns=empty_columns)
    return df_aux

In [20]:
#Lectura de tablas
df_gm_o = pd.read_csv("datasets/Dataset_GM_PR.csv")
df_ca_o = pd.read_csv("datasets/datos_abogados.csv") 
df_naics_o = pd.read_csv("datasets/NAICS_Puerto Rico.csv")

df_gm_o = remove_empty_columns(df_gm_o)
df_ca_o = remove_empty_columns(df_ca_o)
df_naics_o = remove_empty_columns(df_naics_o)

df_gm_o["Email"] = df_gm_o["Email"].str.strip().replace(["Not available"], np.nan)

# Agregar columna que indique de dónde viene el registro
df_gm_o["dataset"] = "gm"
df_ca_o["dataset"] = "ca"
df_naics_o["dataset"] = "naics"

In [21]:
#Agregar columna "is Firm" para diferenciar Firma de Abogados
df_gm_o['is Firm'] = df_gm_o['Name'].apply(lambda x: bool(re.search(src.pattern_firma, str(x), flags=re.IGNORECASE)))

In [22]:
#DataFrame que almacena los registros que son firmas
df_gm_firmas = df_gm_o[df_gm_o["is Firm"]]

#DataFrame filtrado y almacena los registro que son abogados
df_gm_o = df_gm_o[~df_gm_o["is Firm"]]

#Normalización de nombres
df_gm_o["Clean_Name"] = df_gm_o["Name"].apply(lambda x: src.name_normalized(x)).str.lower()
df_ca_o["FULL NAME"] = df_ca_o["FULL NAME"].apply(lambda x: src.name_normalized(x)).str.lower()
df_naics_o["Name_N"] = df_naics_o["Name_N"].apply(lambda x: src.name_normalized(x)).str.lower()

#Reset index
df_gm_o = df_gm_o.reset_index(drop=True)
df_ca_o = df_ca_o.reset_index(drop=True)
df_naics_o = df_naics_o.reset_index(drop=True)

In [23]:
# Columnas importantes: Name, city, state_name, Address, Website,Phone,Google_category,Type,Email,Specialization,Education,ExperiencE
# Columnas a Analizar: zip, state_id, Orig Specialization
# NO Columnas importantes: Google_URL, population,Google_rank,Google_opinions,Google_category,Processed,Name AI

df_gm_o.isna().mean()

Name                   0.000000
Google_URL             0.000000
zip                    0.000000
city                   0.000000
state_id               0.000000
state_name             0.000000
population             0.000000
Address                0.067335
Website                0.659026
Phone                  0.085960
Google_rank            0.193410
Google_opinions        0.177650
Google_category        0.021490
Name len               0.000000
Type                   0.000000
Email                  0.793696
Orig Specialization    0.226361
Specialization         0.226361
Education              0.226361
Experience             0.226361
Processed              0.226361
Name AI                0.226361
dataset                0.000000
is Firm                0.000000
Clean_Name             0.000000
dtype: float64

In [24]:
# Columnas importantes: FULL NAME', 'FNAME', 'LNAME
# Columnas a Analizar: 
# NO Columnas importantes:

df_ca_o.isna().mean()

FULL NAME          0.000000
FNAME              0.000000
LNAME              0.000000
colegiacion        0.092222
rua                0.092593
correo             0.884444
tel_residencial    0.994074
tel_oficina        0.914074
tel_celular        0.957037
otro               0.994444
especialidades     0.635185
Practice Area      0.635185
delegacion         0.001481
State              0.001481
dataset            0.000000
dtype: float64

In [25]:
# Columnas importantes: FULL NAME', 'FNAME', 'LNAME
# Columnas a Analizar: 
# NO Columnas importantes:

df_naics_o.isna().mean()

Name_N                                      0.000000
ZoomInfo Contact ID                         0.000000
Last Name                                   0.000000
First Name                                  0.000000
Middle Name                                 0.517479
Job Title                                   0.127648
Job Title Hierarchy Level                   0.319915
Management Level                            0.686970
Job Start Date                              0.018008
Job Function                                0.462394
Department                                  0.396716
Direct Phone Number                         0.645657
Email Address                               0.253178
Email Domain                                0.253708
Mobile phone                                0.612818
Highest Level of Education                  0.658369
Contact Accuracy Score                      0.000000
Contact Accuracy Grade                      0.000000
ZoomInfo Contact Profile URL                0.

In [26]:
df_ca_o = src2.combine_columns_by_priority(df_ca_o,["tel_celular","tel_residencial","tel_oficina","otro"],"telefono")
df_naics_o = src2.combine_columns_by_priority(df_naics_o,["Mobile phone","Direct Phone Number"], "phone")

In [27]:
df_ca_o["telefono"] = df_ca_o["telefono"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)
df_naics_o["phone"] = df_naics_o["Mobile phone"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)

# Merge por coincidencia exacta de número de contacto

No se obtuvo ninguna coincidencia exacta por número de teléfono entre los 3 datasets.
Se obtuvo 3 coincidencias exactas entre los datasets del colegio de abogados y naics

In [28]:
#Merge entre gm y ca
df_gm_y_df_ca = src2.merge_by_contact_number(df_gm_o, df_ca_o, "Phone", "telefono", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())


                     Clean_Name                 FULL NAME Phone telefono  \
0  orville omar valentin rivera     maria barrera rosario   NaN     None   
1  orville omar valentin rivera     martha berrios rivera   NaN     None   
2  orville omar valentin rivera  catherine catala lasanta   NaN     None   
3  orville omar valentin rivera      luis del valle colon   NaN     None   
4  orville omar valentin rivera     juan encarnacion cruz   NaN     None   

      score  
0  0.367347  
1  0.530612  
2  0.384615  
3  0.333333  
4  0.326531  


In [29]:
#Merge entre gm y naics
df_gm_y_df_naics = src2.merge_by_contact_number(df_gm_o, df_naics_o, "Phone", "phone", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","Phone","phone","score"]].head())

                     Clean_Name                          Name_N Phone phone  \
0  orville omar valentin rivera                    larry suckle   NaN  None   
1  orville omar valentin rivera  alex giovanni rodriguez negron   NaN  None   
2  orville omar valentin rivera                 aura colon sola   NaN  None   
3  orville omar valentin rivera                     judy arroyo   NaN  None   
4  orville omar valentin rivera      yolanda benitez de alegria   NaN  None   

      score  
0  0.300000  
1  0.448276  
2  0.325581  
3  0.205128  
4  0.407407  


In [30]:
#Merge entre ca y naics  -->> **  # Salen 3 registros coincidencia exacta **
df_ca_y_df_naics = src2.merge_by_contact_number(df_ca_o, df_naics_o, "telefono", "phone", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","telefono","phone","score"]].head())               



KeyboardInterrupt: 

In [ ]:
#Merge COMPLETO
df_merged_full = src2.merge_by_contact_number(df_gm_y_df_ca, df_naics_o, "Phone", "phone", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())

# Merge por coincidencia exacta de Email

In [ ]:
#Merge entre gm y ca    -->> ** 5 coincidencias exactas **
df_gm_y_df_ca = src3.merge_by_email(df_gm_o, df_ca_o, "Email", "correo", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","Email","correo","score"]].head())

#Merge entre gm y naics -->> NO COINCIDENCIAS
df_gm_y_df_naics = src3.merge_by_email(df_gm_o, df_naics_o, "Email", "Email Address", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","Phone","phone","score"]].head())

#Merge entre ca y naics  -->> ** 5 coincidencias exactas
df_ca_y_df_naics = src3.merge_by_email(df_ca_o, df_naics_o, "correo", "Email Address", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","correo","Email Address","score"]].head())               

#Merge COMPLETO   -->> NO COINCIDENCIAS
df_merged_full = src3.merge_by_email(df_gm_y_df_ca, df_naics_o, "correo", "Email Address", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())

               Clean_Name               FULL NAME  \
0  lutgardo acevedo lopez  lutgardo acevedo lopez   
1        edil quiles seda        edil quiles seda   
2       omar bonet tirado       omar bonet tirado   
3       onix cintron baez       onix cintron baez   
4     beatriz cay vazquez     beatriz cay vazquez   

                         Email                       correo  score  
0        lacevedolaw@gmail.com        lacevedolaw@gmail.com    1.0  
1       quilesseda@hotmail.com       quilesseda@hotmail.com    1.0  
2          omar.bonet@capr.org          omar.bonet@capr.org    1.0  
3   onix.cintron.law@gmail.com   onix.cintron.law@gmail.com    1.0  
4  beatrizcayvazquez@gmail.com  beatrizcayvazquez@gmail.com    1.0  
Empty DataFrame
Columns: [Clean_Name, Name_N, Phone, phone, score]
Index: []
               FULL NAME                 Name_N                    correo  \
0   jose gonzalez rivera          jose gonzalez   jrg@gonzalezmorales.com   
1  ricardo garcia negron  ricardo ga

# Coincidencias exactas por Nombre

In [ ]:
#Merge entre gm y ca    -->> ** 5 coincidencias exactas **
df_gm_y_df_ca = src3.merge_by_email(df_gm_o, df_ca_o, "Clean_Name", "FULL NAME", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","score"]].head())

#Merge entre gm y naics -->> ** 5 coincidencias exactas **
df_gm_y_df_naics = src3.merge_by_email(df_gm_o, df_naics_o, "Clean_Name", "Name_N", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","score"]].head())

#Merge entre ca y naics  -->> ** 5 coincidencias exactas **
df_ca_y_df_naics = src3.merge_by_email(df_ca_o, df_naics_o, "FULL NAME", "Name_N", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","score"]].head())               

#Merge COMPLETO   -->> NO HAY CONCIDENCIAS  
df_merged_full = src3.merge_by_email(df_gm_y_df_ca, df_naics_o, "Clean_Name", "Name_N", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","score"]].head())

               Clean_Name               FULL NAME  score
0    luis mercado hidalgo    luis mercado hidalgo    1.0
1  lutgardo acevedo lopez  lutgardo acevedo lopez    1.0
2   jorge hernandez lopez   jorge hernandez lopez    1.0
3   angela oquendo negron   angela oquendo negron    1.0
4   manlio arraiza donate   manlio arraiza donate    1.0
         Clean_Name            Name_N  score
0  felipe sotoortiz  felipe sotoortiz    1.0
1     jaime ruberte     jaime ruberte    1.0
2   gilberto oliver   gilberto oliver    1.0
3      jose benitez      jose benitez    1.0
4   francisco ramos   francisco ramos    1.0
                      FULL NAME                        Name_N  score
0   veronica gonzalez rodriguez   veronica gonzalez rodriguez    1.0
1          edwin rivera cintron          edwin rivera cintron    1.0
2             alba lopez arzola             alba lopez arzola    1.0
3       isabel ruberte figueroa       isabel ruberte figueroa    1.0
4  isamillie melendez caraballo  isamillie 

# Merge por coincidencias difusas de nombre

In [41]:
df_gm = df_gm_o[["Clean_Name"]]
df_ca = df_ca_o[["FULL NAME"]]
df_naics = df_naics_o[["Name_N"]]

In [42]:
matches = src.find_fuzzy_matches(df_ca, df_naics,"FULL NAME", "Name_N", 76)

In [43]:
evaluacion = src4.test(matches)


In [44]:
#evaluacion = src4.test_v2(matches)


In [45]:
from sklearn.metrics import confusion_matrix, f1_score

# Suponiendo que tienes:
y_true = evaluacion["actual"]
y_pred = evaluacion["predicted"]

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# F1-score
f1 = f1_score(y_true, y_pred)

# Mostrar resultados
print(f"TP: {tp}")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"F1-score: {f1:.2f}")


TP: 207
TN: 2157
FP: 34
FN: 293
F1-score: 0.56


In [55]:
final = evaluacion[evaluacion["predicted"] == 1]
final.reset_index(drop=True, inplace=True)
final.to_csv()

',name1,name2,score,actual,predicted\n0,ruben lucena quiles,ruben lucena,77.41935483870968,1,1\n1,jose bague soto,jose soto,75.0,0,1\n2,veronica gonzalez rodriguez,veronica gonzalez rodriguez,100.0,1,1\n3,rosa corrada colon,rosa corrada,80.0,1,1\n4,edwin rivera cintron,edwin rivera cintron,100.0,1,1\n5,luisselle quinones maldonado,luisselle quinones,78.26086956521739,1,1\n6,alba lopez arzola,alba lopez arzola,100.0,1,1\n7,jose gonzalez rivera,jose gonzalez,78.78787878787878,1,1\n8,antonio bauza santos,antonio santos,82.35294117647058,1,1\n9,carlos torres velez,carlos velez,77.41935483870968,1,1\n10,jose gonzalez jimenez,jose gonzalez,76.47058823529412,1,1\n11,francisco rodriguez bernier,francisco rodriguez,82.6086956521739,1,1\n12,jose belen rivera,jose rivera,78.57142857142857,1,1\n13,angel flores rivera,angel flores,77.41935483870968,1,1\n14,melissa pellicier ortiz,melissa ortiz,72.22222222222221,0,1\n15,isabel ruberte figueroa,isabel ruberte figueroa,100.0,1,1\n16,isamillie melendez

In [57]:
final.head()

,name1,name2,score,actual,predicted
0,ruben lucena quiles,ruben lucena,77.419355,1,1
1,jose bague soto,jose soto,75.000000,0,1
2,veronica gonzalez rodriguez,veronica gonzalez rodriguez,100.000000,1,1
3,rosa corrada colon,rosa corrada,80.000000,1,1
4,edwin rivera cintron,edwin rivera cintron,100.000000,1,1


In [60]:
final["name2"].duplicated().any()
final[final["name2"].duplicated(keep=False)]



,name1,name2,score,actual,predicted
1,jose bague soto,jose soto,75.000000,0,1
7,jose gonzalez rivera,jose gonzalez,78.787879,1,1
10,jose gonzalez jimenez,jose gonzalez,76.470588,1,1
11,francisco rodriguez bernier,francisco rodriguez,82.608696,1,1
12,jose belen rivera,jose rivera,78.571429,1,1
13,angel flores rivera,angel flores,77.419355,1,1
20,jose santana gonzalez,jose gonzalez,76.470588,1,1
24,francisco santiago rodriguez,francisco rodriguez,80.851064,1,1
31,carlos caban rodriguez,carlos rodriguez,84.210526,1,1
41,jose gonzalez ortiz,jose gonzalez,81.250000,1,1
